# Opinion dynamics: multi-option simulations

Balanced start, one random agent per update sees all other agents (fresh names, permuted labels, shuffled list) and takes the opinion it replies. At most 100 sweeps of N updates, consensus ends a run, opinions without supporters disappear from the prompt. Update step after De Marzo's `LLMs-Opinion-Dynamics` code, extended to q opinions. Writes `data/raw/trajectories.csv` and `data/raw/summary.csv`.

In [ ]:
import os

# before importing vllm, see README
os.environ.setdefault("VLLM_ATTENTION_BACKEND", "FLASH_ATTN")
os.environ.setdefault("VLLM_USE_FLASHINFER_SAMPLER", "0")
os.environ.setdefault("VLLM_DISABLE_FLASHINFER", "1")
os.environ.setdefault("FLASHINFER_DISABLE_VERSION_CHECK", "1")

import json
import random
import re
import string
import time
from pathlib import Path

import numpy as np
import pandas as pd
from vllm import LLM, SamplingParams

In [ ]:
SEED = 7
SEED_OFFSET = 20_000_000

T_MAX_SWEEPS = 100
RECORDS_PER_SWEEP = 10
TEMPERATURE = 0.2
TOP_P = 0.9
MAX_NEW_TOKENS = 16
NAME_LENGTH = 3

# model id, runs advanced together per generate call, vllm settings 
MODELS = {
    "llama3_70b_awq": ("TechxGenus/Meta-Llama-3-70B-Instruct-AWQ", 4,
                       dict(quantization="awq", gpu_memory_utilization=0.92, max_model_len=4096)),
    "llama31_8b_it": ("meta-llama/Llama-3.1-8B-Instruct", 8, dict(gpu_memory_utilization=0.85, max_model_len=4096)),
    "qwen25_7b_it": ("Qwen/Qwen2.5-7B-Instruct", 8, dict(gpu_memory_utilization=0.85, max_model_len=16384)),
    "qwen25_32b_it": ("Qwen/Qwen2.5-32B-Instruct", 4, dict(gpu_memory_utilization=0.90, max_model_len=16384)),
    "gemma4_E4B_it": ("google/gemma-4-E4B-it", 2, dict(gpu_memory_utilization=0.85, max_model_len=16384)),
    "gemma4_31B_dense": ("google/gemma-4-31B-it", 1, dict(gpu_memory_utilization=0.90, max_model_len=8192)),
}
ACTIVE = ["qwen25_7b_it", "qwen25_32b_it"]      # one family per session

# conditions (q, N, runs)
CORE = [25, 50, 100, 200]
PLAN = {
    "gemma4_E4B_it": [(q, N, 3) for q in [3, 10, 50] for N in CORE + [400, 800] if q <= N],
    "gemma4_31B_dense": [(q, N, 3) for q in [3, 10, 50] for N in CORE + [400, 800] if q <= N],
    "qwen25_7b_it": [(q, N, 8) for q in [3, 5, 10] for N in CORE] + [(50, N, 3) for N in [50, 100, 200]]
                    + [(q, N, 3) for q in [3, 10, 50] for N in [400, 800]] + [(2, 200, 10)],
    "qwen25_32b_it": [(q, N, 8) for q in [3, 5, 10] for N in CORE] + [(50, N, 3) for N in [50, 100, 200]]
                     + [(100, 200, 3)] + [(q, N, 3) for q in [3, 10, 50] for N in [400, 800]],
    "llama31_8b_it": [(q, N, 5) for q in [3, 10] for N in CORE] + [(50, N, 3) for N in [50, 100, 200]],
    "llama3_70b_awq": [(q, N, 3) for q in [3, 10] for N in CORE] + [(50, N, 3) for N in [50, 100, 200]],
}

RAW = Path("../data/raw")
RAW.mkdir(parents=True, exist_ok=True)

## Prompt and parser

In [ ]:
def option_labels(q):
    return [a + b for a in string.ascii_lowercase for b in string.ascii_lowercase][:q]


def random_names(n, rng):
    chars = string.ascii_letters + string.digits
    names = set()
    while len(names) < n:
        names.add("".join(rng.choices(chars, k=NAME_LENGTH)))
    return list(names)


def create_prompt(names, opinions):
    lines = ["Below you can see the list of all the other AI agents with the opinion they support.",
             "You must reply with the opinion you want to support.",
             "The opinion must be reported between square brackets.", ""]
    lines += [f"{n}: {o}" for n, o in zip(names, opinions)]
    lines.append("Reply only with the opinion you want to support, between square brackets.")
    return "\n".join(lines)


def parse_reply(text, labels):
    # label between square brackets; without brackets accept a reply that contains exactly one label
    found = re.findall(r"\[([^\]]+)\]", text)
    exact = {c.strip().strip(" .,:;!?'").strip('"') for c in (found if found else [text])} & set(labels)
    if len(exact) == 1:
        return exact.pop()
    if len(exact) > 1:
        return None
    contained = [l for l in labels if re.search(rf"(?<!\w){re.escape(l)}(?!\w)", text.strip())]
    return contained[0] if len(contained) == 1 else None


def chat_format(tok, prompts):
    return [tok.apply_chat_template([{"role": "user", "content": p}], tokenize=False, add_generation_prompt=True)
            for p in prompts]

## One run

In [ ]:
def run_seed(q, N, run):
    return SEED + SEED_OFFSET + 10_000 * q + N + 100_000 * run


def new_run(model_label, q, N, run):
    labels = option_labels(q)
    base, extra = divmod(N, q)
    counts = {l: base + (1 if i < extra else 0) for i, l in enumerate(labels)}    # balanced start
    return {"model_label": model_label, "q": q, "N": N, "run": run, "seed": run_seed(q, N, run),
            "rng": random.Random(run_seed(q, N, run)), "labels": labels, "counts": counts,
            "invalid": 0, "valid": 0, "flips": 0, "reactivations": 0, "consensus_step": None}


def build_prompt(run):
    N, counts, rng, labels = run["N"], run["counts"], run["rng"], run["labels"]
    shown = labels[:]
    rng.shuffle(shown)                          # fresh label permutation
    to_display = dict(zip(labels, shown))
    to_internal = dict(zip(shown, labels))
    names = random_names(N, rng)
    opinions = [l for l in labels for _ in range(counts[l])]
    pairs = list(zip(names, opinions))
    rng.shuffle(pairs)
    i = rng.randint(0, N - 1)                   # focal agent, not shown in the list
    others = [(n, to_display[o]) for k, (n, o) in enumerate(pairs) if k != i]
    return create_prompt([n for n, _ in others], [o for _, o in others]), pairs[i][1], to_internal


def apply_reply(run, own, to_internal, text):
    chosen = parse_reply(text, list(to_internal))    # all q labels stay valid, also extinct ones
    if chosen is None:
        run["invalid"] += 1
        return
    run["valid"] += 1
    chosen = to_internal[chosen]
    if chosen == own:
        return
    if run["counts"][chosen] == 0:
        run["reactivations"] += 1
    run["counts"][own] -= 1
    run["counts"][chosen] += 1
    run["flips"] += 1


def record(run, step, record_type):
    counts, N = run["counts"], run["N"]
    return {"model_label": run["model_label"], "q": run["q"], "N": N, "run": run["run"], "step": step,
            "time": step / N, "record_type": record_type, "counts": json.dumps(counts),
            "leader_share": max(counts.values()) / N,
            "n_active_opinions": sum(c > 0 for c in counts.values()), "invalid_count": run["invalid"]}


def summary(run, model_id):
    counts, N, q = run["counts"], run["N"], run["q"]
    done = run["consensus_step"] is not None
    s = max(counts.values()) / N
    return {"model_label": run["model_label"], "model": model_id, "q": q, "N": N, "run": run["run"],
            "run_seed": run["seed"], "scheduled_horizon_sweeps": T_MAX_SWEEPS, "event_observed": done,
            "consensus_step": run["consensus_step"] if done else np.nan,
            "consensus_time_sweeps": run["consensus_step"] / N if done else np.nan,
            "observed_time_sweeps": run["consensus_step"] / N if done else T_MAX_SWEEPS,
            "counts": json.dumps(counts), "leader_share": s, "potts_order_parameter": (q * s - 1) / (q - 1),
            "n_active_opinions": sum(c > 0 for c in counts.values()),
            "invalid_count": run["invalid"], "valid_count": run["valid"], "flip_count": run["flips"],
            "reactivation_count": run["reactivations"],
            "valid_response_rate": run["valid"] / max(1, run["valid"] + run["invalid"]),
            "temperature": TEMPERATURE, "top_p": TOP_P, "max_new_tokens": MAX_NEW_TOKENS}

In [ ]:
def run_condition(llm, tok, batch_size, model_label, model_id, q, N, n_runs):
    runs = [new_run(model_label, q, N, r) for r in range(n_runs)]
    t_max = T_MAX_SWEEPS * N
    every = max(1, N // RECORDS_PER_SWEEP)
    rows = [record(run, 0, "regular") for run in runs]
    t0 = time.time()
    for step in range(1, t_max + 1):
        active = [run for run in runs if run["consensus_step"] is None]     # consensus is absorbing
        prompts = [build_prompt(run) for run in active]
        for start in range(0, len(active), batch_size):
            batch = prompts[start:start + batch_size]
            outputs = llm.generate(chat_format(tok, [p[0] for p in batch]), sampling, use_tqdm=False)
            for run, (_, own, to_internal), out in zip(active[start:start + batch_size], batch, outputs):
                apply_reply(run, own, to_internal, out.outputs[0].text)
        for run in active:
            if max(run["counts"].values()) == N:
                run["consensus_step"] = step
                if step % every:
                    rows.append(record(run, step, "consensus_event"))    # exact consensus step
        if step % every == 0 or step == t_max:
            rows += [record(run, step, "regular") for run in runs]
        if step % N == 0:
            print(f"{model_label} q={q} N={N}  t={step // N}/{T_MAX_SWEEPS}  "
                  f"mean s={np.mean([max(r['counts'].values()) / N for r in runs]):.3f}  "
                  f"consensus={sum(r['consensus_step'] is not None for r in runs)}/{n_runs}  {time.time() - t0:.0f}s")
    return pd.DataFrame(rows), pd.DataFrame([summary(run, model_id) for run in runs])

## Run

In [ ]:
sampling = SamplingParams(temperature=TEMPERATURE, top_p=TOP_P, max_tokens=MAX_NEW_TOKENS)
for model_label in ACTIVE:
    model_id, batch_size, kwargs = MODELS[model_label]
    llm = LLM(model=model_id, trust_remote_code=True, tensor_parallel_size=1, enforce_eager=True,
              disable_log_stats=True, max_num_seqs=batch_size, **kwargs)
    tok = llm.get_tokenizer()
    for q, N, n_runs in PLAN[model_label]:
        traj, summ = run_condition(llm, tok, batch_size, model_label, model_id, q, N, n_runs)
        for name, df in [("trajectories.csv", traj), ("summary.csv", summ)]:
            df.to_csv(RAW / name, mode="a", header=not (RAW / name).exists(), index=False)
        print(summ[["q", "N", "run", "event_observed", "consensus_time_sweeps", "leader_share", "n_active_opinions"]]
              .to_string(index=False))
    del llm